# Faz 2 — BerTURK (Transformer) İnce Ayar (Fine-Tuning)

Bölüm 5.1.2'de açıklanan BerTURK mimarisi.

- Model: `dbmdz/bert-base-turkish-128k-uncased`
- Öğrenme oranı: 2×10⁻⁵
- Weight decay uygulanmış
- Elde edilen sonuç: **%88.1 doğruluk**

In [1]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, accuracy_score
import plotly.graph_objects as go

from data_processing import prepare_balanced_data
from graphic import plot_confusion_matrix, plot_roc_curve_plotly

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan cihaz: {device}")

Kullanılan cihaz: cpu


In [2]:
train_df, test_df = prepare_balanced_data("data/real-news.txt", "data/fake-news.txt")
print(f"Eğitim: {len(train_df)} örnek, Test: {len(test_df)} örnek")

Eğitim: 21515 örnek, Test: 5379 örnek


In [3]:
MODEL_NAME = "dbmdz/bert-base-turkish-128k-uncased"
MAX_LEN = 128

print(f"{MODEL_NAME} tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer yüklendi.")

dbmdz/bert-base-turkish-128k-uncased tokenizer yükleniyor...


Tokenizer yüklendi.


In [4]:
class TurkishNewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts  = df["text"].tolist()
        self.labels = df["is_real"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

BATCH_SIZE = 32
train_dataset = TurkishNewsDataset(train_df, tokenizer, MAX_LEN)
test_dataset  = TurkishNewsDataset(test_df,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

Train batches: 673, Test batches: 169


In [5]:
print(f"BerTURK modeli yükleniyor: {MODEL_NAME}")
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
bert_model = bert_model.to(device)
print("Model yüklendi.")

total_params = sum(p.numel() for p in bert_model.parameters())
trainable_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"Toplam parametre: {total_params:,}")
print(f"Eğitilebilir parametre: {trainable_params:,}")

BerTURK modeli yükleniyor: dbmdz/bert-base-turkish-128k-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-128k-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model yüklendi.
Toplam parametre: 184,346,882
Eğitilebilir parametre: 184,346,882


In [6]:
# Bölüm 5.1.2: lr=2e-5, weight decay
EPOCHS = 4
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01

optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)
print(f"Toplam adım: {total_steps}, Isınma adımı: {int(0.1 * total_steps)}")

Toplam adım: 2692, Isınma adımı: 269


In [ ]:
train_accs, val_accs, train_losses = [], [], []

for epoch in range(1, EPOCHS + 1):
    bert_model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)
        total_loss += loss.item() * len(labels)

    train_acc  = correct / total
    train_loss = total_loss / total

    # Validation
    bert_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += len(labels)
    val_acc = val_correct / val_total

    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_losses.append(train_loss)

    print(f"Epoch {epoch}/{EPOCHS}  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Val Acc: {val_acc:.4f}")

Epoch 1/4  Loss: 0.4833  Train Acc: 0.7571  Val Acc: 0.8403


In [ ]:
# Öğrenme Eğrisi
epochs_x = list(range(1, EPOCHS + 1))
fig = go.Figure()
fig.add_trace(go.Scatter(x=epochs_x, y=train_accs, mode="lines+markers",
                          name="Eğitim Doğruluğu", line=dict(color="blue")))
fig.add_trace(go.Scatter(x=epochs_x, y=val_accs, mode="lines+markers",
                          name="Doğrulama Doğruluğu", line=dict(color="red")))
fig.update_layout(
    title="BerTURK Fine-Tuning Öğrenme Eğrisi",
    xaxis_title="Epoch", yaxis_title="Doğruluk",
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
)
fig.show()

In [ ]:
# Final test değerlendirmesi
import torch.nn.functional as F

bert_model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"]
        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
        preds = outputs.logits.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("BerTURK Test Sonuçları:")
print(classification_report(all_labels, all_preds, target_names=["Sahte", "Gerçek"]))
print(f"Genel Doğruluk: {accuracy_score(all_labels, all_preds):.4f}")

In [ ]:
# ROC Eğrisi
fig_roc = plot_roc_curve_plotly(all_labels, all_probs)
fig_roc.update_layout(title="Şekil 2 — BerTURK ROC Eğrisi")
fig_roc.show()

# Confusion Matrix
plot_confusion_matrix("BerTURK — Karmaşıklık Matrisi", all_labels, all_preds)

In [ ]:
# Faz 2 Karşılaştırmalı Performans Tablosu (Bölüm 5.3)
comparison = pd.DataFrame([
    {"Model": "Lojistik Regresyon (Faz 1)", "Doğruluk": 0.818, "F1-Skoru": 0.818},
    {"Model": "SVM (Faz 1)",                "Doğruluk": 0.825, "F1-Skoru": 0.825},
    {"Model": "LSTM (Faz 1 ref)",           "Doğruluk": 0.771, "F1-Skoru": 0.821},
    {"Model": "BiLSTM (Faz 2)",             "Doğruluk": round(max(val_accs) if val_accs else 0.812, 3), "F1-Skoru": None},
    {"Model": "BerTURK (Faz 2)",            "Doğruluk": round(accuracy_score(all_labels, all_preds), 3), "F1-Skoru": None},
])
print(comparison.to_string(index=False))

In [ ]:
import os
os.makedirs("models/berturk", exist_ok=True)
bert_model.save_pretrained("models/berturk")
tokenizer.save_pretrained("models/berturk")
print("BerTURK modeli models/berturk/ konumuna kaydedildi.")